In [1]:
import wandb
from src.training.model_trainer import ModelTrainer
from src.preprocessing.experiment_preprocessor import ExperimentPreprocessor
from src.training.objectives import LogisticRegressionObjective, MlpObjective
from src.common.constants import ExperimentPreprocessConfig, ExperimentInfo, Constants as consts
from src.common.wandb_config import WANDB_ENTITY, WANDB_PROJECT
from src.training.artifact_manager import ArtifactManager

In [2]:
from pprint import pprint

expr = "starting_point > 36.0"
preprocess_config = ExperimentPreprocessConfig(
    splits_names=["train", "dev"],
    fraction=0.05,
    use_audio_id_sampling=False,
    use_standardize=False,
    balance_splits_strategy=None,
    remove_by_query=expr,
).get_dict()

experiment_info = ExperimentInfo(
    experiment_name="fft_vs_wavlm_3",
    models=["logistic_regression", "MLP"],
    description="Testing FFT vs WavLM features on Logistic Regression and MLP",
    experiment_preprocess_config=preprocess_config,
).get_dict()

pprint(experiment_info)

{'description': 'Testing FFT vs WavLM features on Logistic Regression and MLP',
 'experiment_name': 'fft_vs_wavlm_3',
 'experiment_preprocess_config': {'balance_splits_strategy': None,
                                  'fraction': 0.05,
                                  'remove_by_query': 'starting_point > 36.0',
                                  'splits_names': ['train', 'dev'],
                                  'use_audio_id_sampling': False,
                                  'use_standardize': False},
 'models': ['logistic_regression', 'MLP']}


In [3]:
wavlm_preprocessor = ExperimentPreprocessor(feat_suffix=consts.wavlm_emb_suffix)

print("Preprocessing WavLM features...")
wavlm_dataset_map = wavlm_preprocessor.preprocess_data(**preprocess_config)

print("Getting Dataloader for training...")
wavlm_dataloaders_map = wavlm_preprocessor.get_dataloaders(wavlm_dataset_map, batch_size=32)
print("Dataloader keys:", wavlm_dataloaders_map.keys())

2026-05-12 23:55:59 | INFO     | FeatureLoader | Using WavLM embeddings suffix
2026-05-12 23:55:59 | INFO     | FeatureLoader | Constructed file path: /Users/mikolajkarapka/Projects/audio-deepfake-detection-uwr/data/collected_data/feature_extracted_wavlm.npy
2026-05-12 23:55:59 | INFO     | FeatureLoader | Constructed file path: /Users/mikolajkarapka/Projects/audio-deepfake-detection-uwr/data/collected_data/feature_extracted.csv
2026-05-12 23:55:59 | INFO     | ExperimentPreprocessor | ExperimentPreprocessor initialized with device: mps and feature suffix: '_wavlm'
2026-05-12 23:55:59 | INFO     | FeatureLoader | Constructed file path: /Users/mikolajkarapka/Projects/audio-deepfake-detection-uwr/data/collected_data/splited_data/feature_extracted_train.csv
2026-05-12 23:55:59 | INFO     | FeatureLoader | Loading metadata from /Users/mikolajkarapka/Projects/audio-deepfake-detection-uwr/data/collected_data/splited_data/feature_extracted_train.csv


Preprocessing WavLM features...


2026-05-12 23:56:06 | INFO     | ExperimentPreprocessor | Removing records from split 'train' using query: starting_point > 36.0
2026-05-12 23:56:06 | INFO     | ExperimentPreprocessor | Sampling 5.0% of data for split 'train'...
2026-05-12 23:56:06 | INFO     | FeatureLoader | Constructed file path: /Users/mikolajkarapka/Projects/audio-deepfake-detection-uwr/data/collected_data/splited_data/feature_extracted_dev.csv
2026-05-12 23:56:06 | INFO     | FeatureLoader | Loading metadata from /Users/mikolajkarapka/Projects/audio-deepfake-detection-uwr/data/collected_data/splited_data/feature_extracted_dev.csv
2026-05-12 23:56:09 | INFO     | ExperimentPreprocessor | Removing records from split 'dev' using query: starting_point > 36.0
2026-05-12 23:56:09 | INFO     | ExperimentPreprocessor | Sampling 5.0% of data for split 'dev'...


Getting Dataloader for training...
Dataloader keys: dict_keys(['train', 'dev'])


In [4]:
fft_preprocess_config = preprocess_config.copy()
fft_preprocess_config["use_standardize"] = True
pprint(fft_preprocess_config)

fft_preprocessor = ExperimentPreprocessor(feat_suffix=consts.fft_emb_suffix)

print("Preprocessing fft features...")
fft_dataset_map = fft_preprocessor.preprocess_data(**fft_preprocess_config)

print("Getting Dataloader for training...")
fft_dataloaders_map = fft_preprocessor.get_dataloaders(fft_dataset_map, batch_size=32)
print("Dataloader keys:", fft_dataloaders_map.keys())

2026-05-12 23:56:09 | INFO     | FeatureLoader | Using FFT embeddings suffix
2026-05-12 23:56:09 | INFO     | FeatureLoader | Constructed file path: /Users/mikolajkarapka/Projects/audio-deepfake-detection-uwr/data/collected_data/feature_extracted_fft.npy
2026-05-12 23:56:09 | INFO     | FeatureLoader | Constructed file path: /Users/mikolajkarapka/Projects/audio-deepfake-detection-uwr/data/collected_data/feature_extracted.csv
2026-05-12 23:56:09 | INFO     | ExperimentPreprocessor | ExperimentPreprocessor initialized with device: mps and feature suffix: '_fft'
2026-05-12 23:56:09 | INFO     | FeatureLoader | Constructed file path: /Users/mikolajkarapka/Projects/audio-deepfake-detection-uwr/data/collected_data/splited_data/feature_extracted_train.csv
2026-05-12 23:56:09 | INFO     | FeatureLoader | Loading metadata from /Users/mikolajkarapka/Projects/audio-deepfake-detection-uwr/data/collected_data/splited_data/feature_extracted_train.csv


{'balance_splits_strategy': None,
 'fraction': 0.05,
 'remove_by_query': 'starting_point > 36.0',
 'splits_names': ['train', 'dev'],
 'use_audio_id_sampling': False,
 'use_standardize': True}
Preprocessing fft features...


2026-05-12 23:56:23 | INFO     | ExperimentPreprocessor | Removing records from split 'train' using query: starting_point > 36.0
2026-05-12 23:56:25 | INFO     | ExperimentPreprocessor | Sampling 5.0% of data for split 'train'...
2026-05-12 23:56:25 | INFO     | ExperimentPreprocessor | Standardizing features for split 'train'...
2026-05-12 23:56:25 | INFO     | FeatureLoader | Constructed file path: /Users/mikolajkarapka/Projects/audio-deepfake-detection-uwr/data/collected_data/splited_data/feature_extracted_dev.csv
2026-05-12 23:56:25 | INFO     | FeatureLoader | Loading metadata from /Users/mikolajkarapka/Projects/audio-deepfake-detection-uwr/data/collected_data/splited_data/feature_extracted_dev.csv
2026-05-12 23:56:29 | INFO     | ExperimentPreprocessor | Removing records from split 'dev' using query: starting_point > 36.0
2026-05-12 23:56:30 | INFO     | ExperimentPreprocessor | Sampling 5.0% of data for split 'dev'...
2026-05-12 23:56:30 | INFO     | ExperimentPreprocessor | Sta

Getting Dataloader for training...
Dataloader keys: dict_keys(['train', 'dev'])


In [5]:
dataloaders_dict = {
    "wavlm": wavlm_dataloaders_map,
    "fft": fft_dataloaders_map,
}

In [6]:

n_trials = 2


run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name=experiment_info["experiment_name"],
    config=experiment_info,
)

model_trainer = ModelTrainer()
artifact_manager = ArtifactManager(experiment_name=experiment_info["experiment_name"])

objectives = [LogisticRegressionObjective, MlpObjective]
objectives_params = {"epochs": 2, "use_pos_weight": True}

for feature_key, dataloaders_map in dataloaders_dict.items():
    train_dataloader = dataloaders_map["train"]
    dev_dataloader = dataloaders_map["dev"]
    for i, objective_cls in enumerate(objectives):
        print(f"Training {objective_cls.__name__} with {feature_key} features...")
        objective = objective_cls(train_loader=train_dataloader, val_loader=dev_dataloader)
        best_params, best_value = model_trainer.optuna_train(
            objective=objective,
            n_trials=n_trials,
            **objectives_params
        )
        print(f"Best params for {objective_cls.__name__} with {feature_key} features: {best_params}, best value: {best_value}")
        file_name = f"{objective_cls.__name__}_{feature_key}_best_params"
        artifact_manager.save_params(
            params=best_params,
            file_name=file_name,
        )
        params_file_path = artifact_manager.get_params_file_path(file_name=file_name)
        run.log_artifact(params_file_path)
        print(f"Saved best params for {objective_cls.__name__} with {feature_key} features to {params_file_path} and logged to wandb.")


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/mikolajkarapka/.netrc.


wandb: Currently logged in as: mkarapka (mkarapka-uniwroc) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


2026-05-12 23:59:20 | INFO     | ModelTrainer | Starting Optuna hyperparameter optimization...
[I 2026-05-12 23:59:20,888] A new study created in memory with name: no-name-aeae02dd-3776-4b09-be53-b309e564d277


Training LogisticRegressionObjective with wavlm features...


  0%|          | 0/2 [00:00<?, ?it/s]

...

  0%|          | 0/2 [00:00<?, ?it/s]

2026-05-13 00:03:06 | INFO     | audio_deepfake.MlpClassifier | Using device: mps
2026-05-13 00:03:27 | INFO     | MlpObjective | Epoch 1/2 - Train Loss: 0.7403, Train Acc: 0.7748, Val Loss: 0.4776, Val Acc: 0.9146, EER: 0.0940
2026-05-13 00:03:47 | INFO     | MlpObjective | Epoch 2/2 - Train Loss: 0.4114, Train Acc: 0.8998, Val Loss: 0.4031, Val Acc: 0.9289, EER: 0.0686
2026-05-13 00:03:47 | INFO     | audio_deepfake.MlpClassifier | Using device: mps


[I 2026-05-13 00:03:47,512] Trial 0 finished with value: 0.06857654452323914 and parameters: {'dropout_rate': 0.30002133693376026, 'n_layers': 2, 'hidden_size_0': 864, 'hidden_size_1': 224, 'lr': 0.00047937662247995376, 'weight_decay': 0.009914131243784647, 'pos_weight': 20.377430412268396}. Best is trial 0 with value: 0.06857654452323914.


2026-05-13 00:04:08 | INFO     | MlpObjective | Epoch 1/2 - Train Loss: 0.9512, Train Acc: 0.6549, Val Loss: 0.7211, Val Acc: 0.8377, EER: 0.1683
2026-05-13 00:04:27 | INFO     | MlpObjective | Epoch 2/2 - Train Loss: 0.5770, Train Acc: 0.8382, Val Loss: 0.4454, Val Acc: 0.8981, EER: 0.0881
2026-05-13 00:04:27 | INFO     | ModelTrainer | Best trial: 0
2026-05-13 00:04:27 | INFO     | ModelTrainer | Best parameters: {'dropout_rate': 0.30002133693376026, 'n_layers': 2, 'hidden_size_0': 864, 'hidden_size_1': 224, 'lr': 0.00047937662247995376, 'weight_decay': 0.009914131243784647, 'pos_weight': 20.377430412268396}
2026-05-13 00:04:27 | INFO     | ModelTrainer | Best value: 0.06857654452323914
2026-05-13 00:04:27 | INFO     | ArtifactManager | Saving params to /Users/mikolajkarapka/Projects/audio-deepfake-detection-uwr/data/params/fft_vs_wavlm_3/MlpObjective_fft_best_params.pkl
2026-05-13 00:04:27 | INFO     | ArtifactManager | Params saved successfully.


[I 2026-05-13 00:04:27,948] Trial 1 finished with value: 0.08805976808071136 and parameters: {'dropout_rate': 0.2621097661385238, 'n_layers': 4, 'hidden_size_0': 704, 'hidden_size_1': 64, 'hidden_size_2': 32, 'hidden_size_3': 32, 'lr': 0.0013561562824772001, 'weight_decay': 0.0002431750485597534, 'pos_weight': 19.825962601435567}. Best is trial 0 with value: 0.06857654452323914.
Best params for MlpObjective with fft features: {'dropout_rate': 0.30002133693376026, 'n_layers': 2, 'hidden_size_0': 864, 'hidden_size_1': 224, 'lr': 0.00047937662247995376, 'weight_decay': 0.009914131243784647, 'pos_weight': 20.377430412268396}, best value: 0.06857654452323914
Saved best params for MlpObjective with fft features to /Users/mikolajkarapka/Projects/audio-deepfake-detection-uwr/data/params/fft_vs_wavlm_3/MlpObjective_fft_best_params.pkl and logged to wandb.
